In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
import seaborn as sns


from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans


import torch
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
import torch.nn as nn
from torch.optim import AdamW
from torchvision.transforms.functional import to_tensor
import torch.nn.functional as F




In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)




In [ ]:
# 2. Create TensorDataset objects

train_dataset = TensorDataset(X_train, y_train)  # <Replace None with your code>
test_dataset  = TensorDataset(X_test, y_test)   # <Replace None with your code>


In [ ]:
# 3. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size= 32, shuffle= True)
test_loader  = DataLoader(test_dataset, batch_size= 32, shuffle= False)




In [ ]:
# 4. Print shape of one batch
for X_batch, y_batch in train_loader:
  print(X_batch.shape)


In [ ]:
# 5. Display sample images




In [ ]:
class NN4Layer(nn.Module):
  def __init__(self, input_dim, hidden_dim, output_dim):
    super(NN4Layer, self).__init__()
    # TODO: Define the first linear layer: input_dim -> hidden_dim
    self.layer1 = nn.Linear(input_dim, hidden_dim)  # <Replace None with your code>

    # TODO: Define the second linear layer: hidden_dim -> hidden_dim
    self.layer2 = nn.Linear(hidden_dim, hidden_dim)  # <Replace None with your code>

    # TODO: Define the output layer: hidden_dim -> 1 (single value for regression)
    self.layer3 = nn.Linear(hidden_dim, hidden_dim)  # <Replace None with your code>

    self.layer4 = nn.Linear(hidden_dim, output_dim)

    # TODO: Define ReLU activation
    self.relu = nn.ReLU()  # <Replace None with your code>

  def forward(self, x):
    # TODO: First hidden layer with ReLU
    a1 = self.relu(self.layer1(x))  # <Replace None with your code>

    # TODO: Second hidden layer with ReLU
    a2 = self.relu(self.layer2(a1))  # <Replace None with your code>

    a3 = self.relu(self.layer3(a2))

    output = self.layer4(a3)



    return output

In [ ]:
# pay extra special for to device and view of batches
# when y is only one columns reshape
# when x is an image reshape (flatten)
# this is for x: X_batch = X_batch.view(X_batch.size(0), -1).to(device)
# this is for y: y_batch = y_batch.view(-1,1).to(device)



def train_one_epoch(model, optimizer, criterion, train_loader, device):
  # TODO: Set the model to training mode
  # <YOUR CODE HERE>
  model.train()
  running_loss = 0.0

  for X_batch, y_batch in train_loader:
    # TODO: Move batch to the selected device
    X_batch = X_batch = X_batch.view(X_batch.size(0), -1).to(device)  # <Replace None with your code>
    y_batch = y_batch.view(-1,1).to(device)  # <Replace None with your code> [HINT: reshape to (-1, 1)]

    # TODO: Forward pass - get model predictions
    outputs = model(X_batch)  # <Replace None with your code>

    # TODO: Compute loss using criterion
    loss = criterion(outputs, y_batch)  # <Replace None with your code>

    # TODO: Backward pass & optimization
    # Step 1: Clear previous gradients
    # <YOUR CODE HERE>
    optimizer.zero_grad()
    # Step 2: Compute gradients (backward pass)
    # <YOUR CODE HERE>
    loss.backward()
    # Step 3: Update model parameters
    # <YOUR CODE HERE>
    optimizer.step()
    running_loss += loss.item()

  # Calculate average loss over all batches
  avg_loss = running_loss / len(train_loader)

  return avg_loss

In [ ]:
# Task 3: Write your validation loop here:
# pay extra special for to device and view of batches
# when y is only one columns reshape
# when x is an image reshape (flatten)
# this is for x: X_batch = X_batch.view(X_batch.size(0), -1).to(device)
# this is for y: y_batch = y_batch.view(-1,1).to(device)



def validate(model, criterion, test_loader, device):
  # TODO: Set the model to evaluation mode
  # <YOUR CODE HERE>
  model.eval()
  running_loss = 0.0

  # TODO: Disable gradient computation using torch.no_grad()
  with torch.no_grad():
    for X_batch, y_batch in test_loader:
      # TODO: Move data to device
      X_batch = X_batch = X_batch.view(X_batch.size(0), -1).to(device)  # <Replace None with your code>
      y_batch = y_batch.view(-1,1).to(device)  # <Replace None with your code> [HINT: reshape to (-1, 1)]

      # TODO: Forward pass - get model predictions
      outputs = model(X_batch) # <Replace None with your code>

      # TODO: Compute loss using criterion
      loss = criterion(outputs, y_batch)  # <Replace None with your code>

      running_loss += loss.item()

  avg_loss = running_loss / len(test_loader)


  return avg_loss

In [ ]:
device = torch.device("cuda" + ":0" if torch.cuda.is_available() else "cpu")

s, c, x, y= X_batch.shape # if its an image
# Model parameters
# TODO: What are input features ? (hint: flatten using channel, height, width)
input_dim = c*x*y# <YOUR CODE HERE>

# TODO: choose number of hidden neurons
hidden_dim = 20# <YOUR CODE HERE>

# TODO: what are the number of classes ?
output_dim = 1# <YOUR CODE HERE>

# TODO: Instantiate model (what are the model class inputs?)
model = NN4Layer(input_dim, hidden_dim, output_dim).to(device)# <YOUR CODE HERE>.to(device)

# TODO: Print the model architecture
print("Model Architecture:\n")
print(model)# <YOUR CODE HERE>)

# Calculate the total number of trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params}")

num_epochs = 20# <YOUR CODE HERE>
# TODO: choose a learning rate (try different values and evaluate results)
learning_rate = 0.0001# <YOUR CODE HERE>

# TODO: Define criterion (loss function) (hint: what loss do we use for multiclass ?)
criterion = nn.MSELoss()# <YOUR CODE HERE>
# TODO: Define optimizer(what is updated during training?)
optimizer = AdamW(model.parameters(), learning_rate)# <YOUR CODE HERE>, # <YOUR CODE HERE>)

In [ ]:
# Task 5: Start training for 20 epochs:
train_losses = []
val_losses = []
val_accuracies = []

print('Starting Training...')
for epoch in range(num_epochs):
    # Train one epoch
    train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)

    # Validate
    val_loss = validate(model, criterion, test_loader, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)


    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

print('Training Complete!')


In [ ]:
# Task 1: Write your code here:
plt.figure(figsize=(7, 5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here: